# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.0229, -0.0927,  0.0416,  0.0378, -0.1320,  0.0707, -0.0441, -0.0426],
        [-0.0083, -0.0677,  0.0232, -0.0132, -0.0539,  0.0363,  0.0154, -0.0934]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[0.0704, 0.0915],
        [0.3353, 0.5586],
        [0.4701, 0.0490],
        [0.3637, 1.0039],
        [0.0680, 0.4292]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[-0.0173,  0.0700],
        [ 0.2312,  0.2760],
        [ 0.2374,  0.4548],
        [ 0.1273,  0.1862],
        [ 0.1778,  0.6018]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [7]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    @torch.inference_mode()
    def forward(self, x):
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[ 0.5582, -0.3438],
        [ 0.5035, -1.2915],
        [-0.5675, -0.6704],
        [ 0.7045,  0.3180],
        [ 0.2915,  0.1119]])
Output inference_mode method:
 tensor([[ 0.5582, -0.3438],
        [ 0.5035, -1.2915],
        [-0.5675, -0.6704],
        [ 0.7045,  0.3180],
        [ 0.2915,  0.1119]])
Output no_grad context manager:
 tensor([[ 0.5582, -0.3438],
        [ 0.5035, -1.2915],
        [-0.5675, -0.6704],
        [ 0.7045,  0.3180],
        [ 0.2915,  0.1119]])
Output inference_mode context manager:
 tensor([[ 0.5582, -0.3438],
        [ 0.5035, -1.2915],
        [-0.5675, -0.6704],
        [ 0.7045,  0.3180],
        [ 0.2915,  0.1119]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [8]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [9]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.3310, 0.0923, 3.1094, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.0000, 0.6804, 0.9700, 0.2589],
        [0.1655, 0.0461, 1.5547, 0.0000]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [10]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [11]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([ # полезно в агрегации признаков, например
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        result = []
        for layer in self.layers:
            x = layer(x)
            result.append(x)
        #print(result)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

for name, param in ml_model.named_parameters():
    print(name, param.shape)

nn.ModuleList output shape: torch.Size([5, 2])
layers.0.weight torch.Size([8, 4])
layers.0.bias torch.Size([8])
layers.2.weight torch.Size([2, 8])
layers.2.bias torch.Size([2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [12]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])



## Работа на семинаре

### LSTM

![lstm](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/LSTM.png?raw=1)

In [13]:
#x1 = [1, 2, 3, 4, 5, 6]
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Self, Tuple

class LSTMCell(nn.Module): # если нужна какая-то память о предыдущих событиях (пример: time-series)
    def __init__(self, input_size: int, hidden_size: int, bias: bool = False) -> None: # полезно использовать type hints
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.x_gates = nn.Linear(input_size, hidden_size*4, bias=bias)
        self.h_gates = nn.Linear(hidden_size, hidden_size*4, bias=bias)

    # очень не рекомендуется использовать if else в forward, лучше уж torch.branch
    # или в init всё выбрать (пример: loss-функция передается как строка, в forward как функция должна идти)
    def forward(self, x: torch.Tensor, h_prev: torch.Tensor, c_prev) -> Tuple[torch.Tensor, torch.Tensor]:
        '''
        Input:
            x_t: [batch_size, input_size]
            h_{t-1}: [batch_size, hidden_size]
            c_{t-1}: [batch_size, hidden_size]

        Output:
            h_t: [batch_size, hidden_size]
            c_t: [batch_size, hidden_size]
        '''
        gates_out = self.x_gates(x) + self.h_gates(h_prev)

        f_gate, c_gate, i_gate, o_gate = torch.chunk(gates_out, 4, -1)

        f_gate = F.sigmoid(f_gate)
        c_gate = F.tanh(c_gate)
        i_gate = F.sigmoid(i_gate)
        o_gate = F.sigmoid(o_gate)

        ci_gate = c_gate * i_gate
        c_t = f_gate * c_prev + ci_gate
        h_t = F.tanh(c_t) * o_gate

        return c_t, h_t


x = torch.rand(2, 2)
c = torch.rand(2, 4)
h = torch.rand(2, 4)

lstm = LSTMCell(2, 4)

h, c = lstm(x, h, c)
print(h)
print(c)

h = torch.zeros(2,4)
c = torch.zeros(2, 4)

x_lst = [torch.rand(2,2) for _ in range(5)]

for x in x_lst:
    h, c = lstm(x, h, c)

# out = out(c)
# loss= mse(tagret, out)
# loss.backward() # автоматически

tensor([[ 0.5148,  0.4917,  0.3788,  0.2035],
        [ 0.6810, -0.0019,  0.5994,  0.1542]], grad_fn=<AddBackward0>)
tensor([[ 0.2162,  0.2604,  0.1622,  0.1141],
        [ 0.2124, -0.0012,  0.1619,  0.0996]], grad_fn=<MulBackward0>)


### Inception

![inception](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/inception.png?raw=1)

### SE

![se](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/SqueezeAndExcite.png?raw=1)

### Selective Kernel

![selective](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/SelectiveKernel.png?raw=1)


### PatchMerger

![patchmerger](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/PatchMerger.png?raw=1)


## Homework

2 задания:
1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).

## ResNet Block (0.1 балл)

![Resnet](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/ResBlock.png?raw=1)

https://arxiv.org/pdf/1512.03385

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Self, Tuple

class ResidualBlock(nn.Module):
    def __init__(self, input_size: int) -> None:
        super().__init__()

        self.layer1 = nn.Conv1d(input_size, input_size, 3, padding=1)
        self.layer2 = nn.Conv1d(input_size, input_size, 5, padding=2)

    def forward(self, x: Tensor) -> Tensor:
        fx = self.layer1(x)
        fx = self.layer2(fx)
        return x + fx

x = torch.rand(5, 4)
model = ResidualBlock(5)
for i in range(3):
    x = model(x)
    print(x)

tensor([[ 0.8938,  0.7203,  0.4921,  0.7562],
        [ 0.8100,  0.5063,  0.8508,  0.4467],
        [ 0.5037,  0.5248,  0.4735,  0.3115],
        [ 0.5619,  1.0016, -0.0103,  0.6732],
        [ 0.7483,  0.5629,  0.2123,  1.0329]], grad_fn=<AddBackward0>)
tensor([[ 0.8519,  0.7453,  0.4756,  0.7735],
        [ 0.8494,  0.5633,  1.0911,  0.4476],
        [ 0.1424,  0.4297,  0.2794,  0.3182],
        [ 0.6260,  1.0455, -0.1625,  0.4802],
        [ 0.9747,  0.5885,  0.0956,  1.1892]], grad_fn=<AddBackward0>)
tensor([[ 0.8215,  0.7585,  0.4832,  0.8219],
        [ 0.8667,  0.6368,  1.3426,  0.4670],
        [-0.2223,  0.2830,  0.1219,  0.3183],
        [ 0.7285,  1.0831, -0.3047,  0.2774],
        [ 1.2079,  0.6316, -0.0365,  1.3618]], grad_fn=<AddBackward0>)


## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/DepthWiseConv.png?raw=1)

https://arxiv.org/pdf/1610.02357

In [15]:
import torch
import torch.nn as nn

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()

        # D_f × D_f × M -> D_g × D_g × M
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size,
            stride=stride, padding=padding, groups=in_channels, bias=False
        )

        # D_g × D_g × M -> D_g × D_g × N
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, 1, stride=1, padding=0, bias=False
        )

    def forward(self, x):
        """
            Input: Tensor[D_f, D_f, M]
            Output: Tensor[D_g, D_g, N]
        """
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

x = torch.randn(2, 3, 32, 32)
model = SeparableConv2d(3, 64, kernel_size=3, stride=1, padding=1)
y = model(x)
print(f"Вход: {x.shape}")
print(f"Выход: {y.shape}")

Вход: torch.Size([2, 3, 32, 32])
Выход: torch.Size([2, 64, 32, 32])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [16]:
from typing import Optional
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, d):
        super().__init__()

        self.W_align = nn.Linear(d, d, bias=False)
        self.W_value = nn.Linear(d, d, bias=False)
        self.W_query = nn.Linear(d, d, bias=False)

    def forward(self, query: torch.Tensor, key: torch.Tensor) -> torch.Tensor:
        '''
            Input:
                query: [B, d]
                key: [B, L, d]

            Output:
                out: [B, d]
        '''
        transformed_query = self.W_align(query).unsqueeze(-1) # [B, d, 1]
        score = (key @ transformed_query).squeeze(-1) # [B, L]
        att = F.softmax(score, dim=1) # [B, L]
        context = (att.unsqueeze(1) @ key).squeeze(1) # [B, d]
        out = F.tanh(self.W_value(context) + self.W_query(query)) # [B, d]
        return out

B = 4
d = 128
L = 10

query = torch.randn(B, d)
key = torch.randn(B, L, d)

attention = VanillaAttention(d)
output = attention(query, key)

print(f"Форма query:  {query.shape}")
print(f"Форма key:    {key.shape}")
print(f"Форма output: {output.shape}")

Форма query:  torch.Size([4, 128])
Форма key:    torch.Size([4, 10, 128])
Форма output: torch.Size([4, 128])


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()


    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> torch.Tensor:
        """
        Input:
            query: [B, L_q, d_k]
            key:   [B, L_k, d_k]
            value: [B, L_k, d_k]

        Output:
            output: [B, L_q, d_k]
        """
        d_k = query.size(-1)
        scores = (query @ key.transpose(-2, -1)) / np.sqrt(d_k)
        attention_weights = F.softmax(scores, dim=-1)
        output = attention_weights @ value

        return output

batch_size = 2
seq_len_q = 5   # L_q
seq_len_k = 7   # L_k
d_k = 64

Q = torch.randn(batch_size, seq_len_q, d_k)
K = torch.randn(batch_size, seq_len_k, d_k)
V = torch.randn(batch_size, seq_len_k, d_k)

attention = ScaledDotProductAttention()
output = attention(Q, K, V)

print(f"Форма Q: {Q.shape}")
print(f"Форма K: {K.shape}")
print(f"Форма V: {V.shape}")
print(f"Форма выхода: {output.shape}")

Форма Q: torch.Size([2, 5, 64])
Форма K: torch.Size([2, 7, 64])
Форма V: torch.Size([2, 7, 64])
Форма выхода: torch.Size([2, 5, 64])


## Multihead Attention (0.1 балл)

![MultiheadAttention](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/MultiheadAttention.webp?raw=1)

https://arxiv.org/abs/1706.03762


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d, num_heads):
        super().__init__()
        assert d % num_heads == 0

        self.num_heads = num_heads
        self.d_k = d // num_heads

        self.W_q = nn.Linear(d, d, bias=False)
        self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False)
        self.W_o = nn.Linear(d, d, bias=False)

        self.attention = ScaledDotProductAttention()

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> torch.Tensor:
        batch_size = query.size(0)

        # 1. Linear
        Q = self.W_q(query)   # [B, L_q, d]
        K = self.W_k(key)     # [B, L_k, d]
        V = self.W_v(value)   # [B, L_k, d]

        # 2. Разделяем на heads
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k)
        K = K.view(batch_size, -1, self.num_heads, self.d_k)
        V = V.view(batch_size, -1, self.num_heads, self.d_k)

        # 3. Переставляем для внимания
        Q = Q.transpose(1, 2)  # [B, num_heads, L_q, d_k]
        K = K.transpose(1, 2)  # [B, num_heads, L_k, d_k]
        V = V.transpose(1, 2)  # [B, num_heads, L_k, d_k]

        # 4. Внимание на всех головах сразу
        att = self.attention(Q, K, V)  # [B, num_heads, L_q, d_k]

        # 5. Объединяем heads
        att = att.transpose(1, 2).contiguous()  # [B, L_q, num_heads, d_k]
        att = att.view(batch_size, -1, self.num_heads * self.d_k)  # [B, L_q, d]

        # 6. Финальный слой
        return self.W_o(att)

batch_size = 2
seq_len_q = 5
seq_len_k = 7
d_model = 512
num_heads = 8

query = torch.randn(batch_size, seq_len_q, d_model)
key = torch.randn(batch_size, seq_len_k, d_model)
value = torch.randn(batch_size, seq_len_k, d_model)

mha = MultiHeadAttention(d_model, num_heads)

output = mha(query, key, value)

print(f"Форма Q: {query.shape}")
print(f"Форма K: {key.shape}")
print(f"Форма V: {value.shape}")
print(f"Форма выхода: {output.shape}")

Форма Q: torch.Size([2, 5, 512])
Форма K: torch.Size([2, 7, 512])
Форма V: torch.Size([2, 7, 512])
Форма выхода: torch.Size([2, 5, 512])


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/TransformerEncoder.png?raw=1)


https://arxiv.org/abs/1706.03762

In [19]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d: int, num_heads: int, dim_mlp: int, dropout: float = 0.1):
        """
        Args:
            d: размерность модели (входная и выходная)
            num_heads: количество голов внимания
            dim_feedforward: размерность внутреннего слоя FFN
            dropout: вероятность dropout
        """
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.self_attn = MultiHeadAttention(d, num_heads)
        self.norm2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(
            nn.Linear(d, dim_mlp),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_mlp, d),
            nn.Dropout(dropout)
        )

    def forward(self, t_n: torch.Tensor) -> torch.Tensor:
        """
        Input:  T_N     [B, L, d_model]

        Output: T_{N+1} [B, L, d_model]
        """
        normed = self.norm1(t_n)
        attn_out = self.self_attn(normed, normed, normed)
        t_n = t_n + attn_out

        normed = self.norm2(t_n)
        mlp_out = self.mlp(normed)
        t_n = t_n + mlp_out

        return t_n


batch_size, seq_len, d = 2, 10, 512
num_heads, dim_mlp = 8, 2048
dropout = 0.1

encoder_layer = TransformerEncoderLayer(d, num_heads, dim_mlp, dropout)

x = torch.randn(batch_size, seq_len, d)

output = encoder_layer(x)

print(f"Входная форма: {x.shape}")
print(f"Выходная форма: {output.shape}")

Входная форма: torch.Size([2, 10, 512])
Выходная форма: torch.Size([2, 10, 512])


## MLP Mixer (0.1 балл)


![MLPMixer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/MLPMixer.png?raw=1)


https://arxiv.org/abs/2105.01601

In [20]:
class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches: int, dim: int, tokens_mlp_dim: int, channels_mlp_dim: int):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        self.mlp1 = nn.Sequential(
            nn.Linear(num_patches, tokens_mlp_dim),
            nn.GELU(),
            nn.Linear(tokens_mlp_dim, num_patches)
        )

        self.norm2 = nn.LayerNorm(dim)

        self.mlp2 = nn.Sequential(
            nn.Linear(dim, channels_mlp_dim),
            nn.GELU(),
            nn.Linear(channels_mlp_dim, dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Input: [B, num_patches, dim]
        Output: [B, num_patches, dim]
        """
        residual = x
        x_norm = self.norm1(x)  # [B, num_patches, dim]
        x_t = x_norm.transpose(1, 2)
        x_t = self.mlp1(x_t)  # [B, dim, num_patches]
        x = x_t.transpose(1, 2)
        x = x + residual

        residual = x
        x_norm = self.norm2(x)  # [B, num_patches, dim]
        x = self.mlp2(x_norm)  # [B, num_patches, dim]
        x = x + residual

        return x

batch_size = 4
num_patches = 196  # например, 14x14 патчей
dim = 512          # размерность каналов
tokens_mlp_dim = 256
channels_mlp_dim = 1024

# Создаем блок
mixer_block = MLPMixerBlock(
    num_patches=num_patches,
    dim=dim,
    tokens_mlp_dim=tokens_mlp_dim,
    channels_mlp_dim=channels_mlp_dim
)

# Входной тензор
x = torch.randn(batch_size, num_patches, dim)

# Прямой проход
output = mixer_block(x)

print(f"Входная форма: {x.shape}")      # [4, 196, 512]
print(f"Выходная форма: {output.shape}") # [4, 196, 512]

Входная форма: torch.Size([4, 196, 512])
Выходная форма: torch.Size([4, 196, 512])


## ConvMixer (0.1 балл)

![ConvMixer](https://github.com/v3code/MTAI2026-Deep-learning-frameworks-seminars/blob/main/seminar_2/assets/ConvMixer.png?raw=1)


https://arxiv.org/abs/2201.09792

In [21]:
class ConvMixer(nn.Module):

    def __init__(self, dim, kernel_size):
        super().__init__()

        self.depthwise = nn.Conv2d(
            dim, dim, kernel_size,
            padding=kernel_size // 2, groups=dim
        )

        self.act1 = nn.ReLU()
        self.norm1 = nn.BatchNorm2d(dim)

        self.pointwise = nn.Conv2d(
            dim, dim, 1
        )

        self.act2 = nn.ReLU()
        self.norm2 = nn.BatchNorm2d(dim)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # [B, dim, H, W] -> [B, dim, H, W]
        residual = x

        x = self.depthwise(x)
        x = self.act1(x)
        x = self.norm1(x)

        x = x + residual

        x = self.pointwise(x)
        x = self.act2(x)
        x = self.norm2(x)

        return x

batch_size = 4
dim = 256
kernel_size = 7
height = width = 16

conv_mixer = ConvMixer(dim=dim, kernel_size=kernel_size)

x = torch.randn(batch_size, dim, height, width)

output = conv_mixer(x)

print(f"Входная форма: {x.shape}")      # [4, 256, 16, 16]
print(f"Выходная форма: {output.shape}") # [4, 256, 16, 16]

Входная форма: torch.Size([4, 256, 16, 16])
Выходная форма: torch.Size([4, 256, 16, 16])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

**Ответ:** MLPMixer и ConvMixer работают так же эффективно, как Multihead Attention, потому что все три архитектуры выполняют одни и те же две фундаментальные операции: смешивание пространственной информации и смешивание информации по каналам, но разными способами.

**Общая формула:**  
Для входного тензора $X \in ℝ^{S×C}$ (S — пространственные позиции, C — каналы):
1. $Y = X + \text{Mix}_S(\text{Norm}(X))$ (пространственное смешивание)
2. $Z = Y + \text{Mix}_C(\text{Norm}(Y))$ (канальное смешивание)

Итого: $X + \text{Mix}_S(\text{Norm}(X)) + \text{Mix}_C(\text{Norm}(X + \text{Mix}_S(\text{Norm}(X))))$

**Преимущества и недостатки:**

Multihead Attention:
* Преимущества: Гибкость, глобальный контекст
* Недостатки: Квадратичная сложность $O(S^2)$, много данных  

MLPMixer:
* Преимущества: Простота, эффективнее MHA  
* Недостатки: Много параметров, требует позиционное кодирование  

ConvMixer:
* Преимущества: Быстрый, мало параметров, нет позиционного кодирования
* Недостатки: Локальность сверток (ограниченное поле зрения)